# Bản mini SmolVLA: fine-tune -> chấm -> mở hộp đen, ở máy nhà (~5 phút)

Đọc `mini_local.ipynb` (ACT) trước: cùng dataset, cùng cách chạy. Ở đây chỉ nói chỗ **khác** của SmolVLA:

| | ACT | SmolVLA |
|---|---|---|
| Kích thước | ~52 triệu tham số, train từ đầu | ~450 triệu, **học sẵn** (`lerobot/smolvla_base`) rồi fine-tune |
| Đọc ảnh | ResNet18 | SmolVLM2 (mô hình ảnh + chữ), bị **khoá** khi fine-tune |
| Đọc lệnh chữ | Không | Có: câu "Pick up the red box..." được tách thành token đưa vào mạng |
| Ra action | Transformer đoán thẳng 50 action | "Action expert" khử nhiễu dần (flow matching, 10 lần) ra 50 action |
| Đoán lại sau | 20 action | 50 action |
| Camera | top, wrist | camera1, camera2 (camera3 thiếu thì bù ảnh trống) |

Mạng mini chỉ fine-tune 100 step trên 3 ván nên **sẽ gắp hỏng**: mục đích là thấy đường chạy, không phải ra mạng giỏi. Bản thật:
notebook Colab mục 8 (~2 000 ván, 20 000 step).

**Chạy:** VS Code, kernel `D:/vla/venv-lerobot/Scripts/python.exe` (đã cài `lerobot[smolvla]`), chạy lần lượt từng ô.

In [ ]:
# ===== Ô 0: chuẩn bị (giống mini_local.ipynb, xem chú thích chi tiết ở đó) =====
import os, shutil, subprocess, sys
from pathlib import Path

import sim_engine
REPO = Path(sim_engine.__file__).parent          # .../VLA/sim-engine
MINI = Path('D:/vla/mini')                       # dataset + checkpoint của bản mini
PY = sys.executable                              # python của kernel (venv-lerobot)

def run(module, *args, keep=lambda line: True):
    """Chạy `python -m module args...`, in output từng dòng (bỏ cảnh báo, log mã hoá video, thanh tiến độ)."""
    env = {**os.environ, 'PYTHONIOENCODING': 'utf-8'}
    p = subprocess.Popen([PY, '-m', module, *map(str, args)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, encoding='utf-8', errors='replace', env=env)
    for line in p.stdout:
        noise = 'warn' in line.lower() or line.startswith(('Svt[', '[mp4', 'Map:'))
        if not noise and keep(line):
            print(line, end='')
    if p.wait() != 0:
        raise RuntimeError(f'{module} lỗi (mã {p.returncode})')

# SmolVLA cần thư viện transformers (cài bằng: pip install "lerobot[smolvla]==0.7.0"). Thiếu thì dòng import này báo lỗi.
import transformers, torch
print('transformers', transformers.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'không có')

# --- Bản học sẵn smolvla_base (~870 MB) ---
# Trên Colab (Linux) ghi thẳng policy.path=lerobot/smolvla_base là được. Trên Windows LeRobot đổi '/' thành '\\' rồi không nhận ra
# đây là tên trên Hugging Face (lỗi "Repo id must use alphanumeric chars ... 'lerobot\\smolvla_base'"), nên tải về thư mục riêng
# một lần rồi trỏ tới thư mục đó.
from huggingface_hub import snapshot_download
MODEL = Path('D:/vla/hf-models/smolvla_base')
if not (MODEL / 'model.safetensors').exists():
    snapshot_download('lerobot/smolvla_base', local_dir=MODEL)   # chỉ tải lần đầu
print('smolvla_base:', MODEL)

## 1. Dataset: dùng lại 3 ván của bản mini ACT

Cùng định dạng LeRobotDataset, nên dataset ghi cho ACT dùng thẳng cho SmolVLA, không phải ghi lại. Chưa chạy `mini_local.ipynb` thì
ô dưới tự ghi 3 ván.

In [ ]:
# ===== Ô 1: dataset =====
DS = MINI / 'datasets' / 'so101_mini'
if not (DS / 'meta' / 'info.json').exists():        # chưa có thì ghi 3 ván (thầy min-jerk, ~1 phút)
    shutil.rmtree(DS, ignore_errors=True)
    run('sim_engine.learning.record_dataset', '--episodes', 3, '--cameras', 'top,wrist', '--root', DS, '--repo-id', 'local/so101_mini')
print('dataset:', DS)

## 2. Xem bản học sẵn trước khi fine-tune

`config.json` của smolvla_base cho biết mạng mong đợi đầu vào gì. Ba camera tên `camera1/2/3` (lúc học sẵn, mỗi bộ dữ liệu cộng đồng đặt
tên camera khác nhau nên được quy về tên chung). Dataset của ta có `top`, `wrist` nên phải đổi tên khi train (`rename_map`).

In [ ]:
# ===== Ô 2: đọc config của bản học sẵn =====
import json
cfg = json.loads((MODEL / 'config.json').read_text())
print('đầu vào :', {k: v['shape'] for k, v in cfg['input_features'].items()})   # state 6 số + 3 ảnh 256x256
print('đầu ra  :', {k: v['shape'] for k, v in cfg['output_features'].items()})  # action 6 số (5 khớp + kẹp), giống ACT
# Các tham số quan trọng (đọc kèm learning/configs/smolvla_finetune.yaml):
for k in ['chunk_size',            # đoán 50 action một lần, như ACT
          'n_action_steps',        # làm hết 50 rồi mới đoán lại (ACT của ta: 20)
          'num_steps',             # số lần khử nhiễu khi ra action (flow matching): nhiều hơn = mịn hơn nhưng chậm hơn
          'freeze_vision_encoder', # True: phần nhìn (SmolVLM2) giữ nguyên, không học thêm -> train nhẹ, không làm hỏng cái đã học sẵn
          'train_expert_only',     # True: chỉ train "action expert" (phần ra action) + vài lớp chiếu
          'tokenizer_max_length',  # câu lệnh dài tối đa 48 token
          'optimizer_lr']:         # lr 1e-4, preset có sẵn lịch giảm dần (cosine) - khác ACT phải tự đặt
    print(f'  {k:24s} {cfg.get(k)}')

## 3. Fine-tune 100 step

Cùng `learning/train.py`, khác file YAML. Mỗi step chậm hơn ACT nhiều (~1,2 s so với ~0,3 s trên GTX 1650) vì mạng to hơn 9 lần,
nhưng bộ nhớ vẫn nhỏ (~1,7 GB) nhờ khoá phần nhìn. Log có thêm vài cột `losses_after_*`: loss trước / sau khi bỏ phần đệm (action
vượt cuối ván, chiều action đệm thêm cho đủ 32); cột đáng xem vẫn là `loss`.

In [ ]:
# ===== Ô 3: fine-tune =====
OUT = MINI / 'outputs' / 'smolvla_mini'
shutil.rmtree(OUT, ignore_errors=True)
run('sim_engine.learning.train',
    REPO / 'learning/configs/smolvla_finetune.yaml',   # đọc file này: có rename_map top -> camera1, wrist -> camera2
    f'--policy.path={MODEL}',                          # thay 'lerobot/smolvla_base' bằng thư mục đã tải (lỗi Windows ở ô 0)
    '--dataset.repo_id=local/so101_mini', f'--dataset.root={DS}', f'--output_dir={OUT}', '--job_name=smolvla_mini',
    '--steps=100',                                     # bản thật 20 000
    '--batch_size=2',                                  # GTX 1650 4 GB; bản thật 32 trên L4
    '--save_freq=100', '--log_freq=20', '--num_workers=0',
    keep=lambda line: 'step:' in line or 'Error' in line or 'lerobot-train' in line)
# Checkpoint giống ACT: config.json (đầu vào đã đổi tên camera), model.safetensors, bộ chuẩn hoá. Có thêm bộ tách token câu lệnh.
print(sorted(p.name for p in (OUT / 'checkpoints/last/pretrained_model').iterdir()))

## 4. Chấm 1 seed

Cùng `eval_policy.py` như ACT. Checkpoint đòi camera `camera1`, `camera2`: `sim_setup.CAMERAS` có sẵn `camera1 -> topcam`,
`camera2 -> arm_cam` nên dựng đúng ảnh. Mỗi ván chậm hơn ACT (~18 s so với ~5 s cho 500 bước trên GTX 1650).

In [ ]:
# ===== Ô 4: chấm =====
run('sim_engine.learning.eval_policy', '--policy', OUT / 'checkpoints/last/pretrained_model', '--seeds', '0')

## 5. Mở hộp đen: câu lệnh đi vào mạng thế nào, mạng đoán lại khi nào

Giống ô 4 của `mini_local.ipynb`, thêm hai thứ chỉ VLA có: câu lệnh chữ thành token, và thời gian một lần đoán (khử nhiễu 10 lần).

In [ ]:
# ===== Ô 5: tự viết vòng lặp, nhìn vào bên trong =====
import time
import numpy as np
from lerobot.common.control_utils import predict_action
from lerobot.policies import prepare_observation_for_inference   # bước đầu của predict_action: numpy -> tensor GPU, thêm chiều batch
from sim_engine.learning import sim_setup
from sim_engine.learning.eval_policy import load_policy

policy, pre, post = load_policy(OUT / 'checkpoints/last/pretrained_model', 'cuda')
# Đầu vào mạng đòi: state + camera1 + camera2 + camera3. spec_from_features bỏ camera3 (ta không có) -> cameras = ('camera1', 'camera2')
cameras, env_state = sim_setup.spec_from_features(policy.config.input_features)
print('mạng đòi:', list(policy.config.input_features), '-> ta dựng:', cameras)
# Đếm tham số: tổng, và phần được train (phần nhìn bị khoá nên ít hơn nhiều)
total = sum(p.numel() for p in policy.parameters())
trainable = sum(p.numel() for p in policy.parameters() if p.requires_grad)
print(f'tham số: {total / 1e6:.0f} triệu, được train: {trainable / 1e6:.0f} triệu')

task = sim_setup.TASKS['box'].format(color='red')
env, options = sim_setup.make_env(0, 'box', 0.04, images=True)
obs, info = env.reset(seed=0, options=options)
policy.reset()

# --- Câu lệnh chữ đi vào mạng thế nào ---
frame = sim_setup.to_frame(env, obs, np.zeros(7), task, cameras, env_state)
observation = {k: v for k, v in frame.items() if k.startswith('observation.')}
batch = pre(prepare_observation_for_inference(dict(observation), torch.device('cuda'), task, None))
print('sau tiền xử lý:', {k: tuple(v.shape) for k, v in batch.items() if hasattr(v, 'shape')})
tokens = batch['observation.language.tokens'][0]                     # câu lệnh -> dãy số (token), đệm cho đủ 48
mask = batch['observation.language.attention_mask'][0].bool()        # True = token thật, False = đệm
tok = policy.model.vlm_with_expert.processor.tokenizer                # bộ tách token của SmolVLM2
print('token:', tokens[mask].tolist())
print('đọc lại:', repr(tok.decode(tokens[mask])))

# --- Vòng kín 60 bước ---
for t in range(60):
    frame = sim_setup.to_frame(env, obs, np.zeros(7), task, cameras, env_state)
    observation = {k: v for k, v in frame.items() if k.startswith('observation.')}
    queued = len(policy._queues['action'])            # hàng đợi action của SmolVLA (ACT: policy._action_queue)
    t0 = time.time()
    action = predict_action(observation, policy, torch.device('cuda'), pre, post, use_amp=False, task=task)
    dt = (time.time() - t0) * 1000
    obs, _, _, _, info = env.step(sim_setup.to_env_action(action.cpu().numpy()))
    if queued == 0 or t % 15 == 0:
        # queued == 0: bước này mạng thật sự chạy (đọc ảnh + câu lệnh, khử nhiễu 10 lần) -> chậm; các bước khác chỉ lấy action có sẵn
        print(f'bước {t:2d} | hàng đợi {queued:2d} | {dt:6.1f} ms | action {action.cpu().numpy().ravel().round(2)} '
              f'| đầu kẹp cách hộp {np.linalg.norm(info["ee_pos"] - info["box_pos"]) * 100:.1f} cm')
env.close()
# Đọc kết quả (chạy thử trên GTX 1650): bước 0 và 50 mất ~1 000 ms (mạng chạy thật: đọc ảnh + câu lệnh, khử nhiễu 10 lần), các bước
# giữa ~6 ms (lấy action có sẵn trong hàng đợi). Mới 100 step trên 3 ván mà tay đã đi về phía hộp (33 cm -> ~10 cm): đó là phần
# học sẵn của smolvla_base. Chưa đủ để gắp; tham số 450 triệu nhưng chỉ ~100 triệu được train.

## Đọc gì tiếp

| File | Đọc để hiểu |
|---|---|
| `learning/configs/smolvla_finetune.yaml` | Tham số fine-tune, `rename_map` |
| `learning/sim_setup.py` (`CAMERAS`, `spec_from_features`) | Camera1/2 của SmolVLA trỏ về camera nào trong sim |
| LeRobot `policies/smolvla/modeling_smolvla.py` (trong `D:/vla/venv-lerobot/Lib/site-packages/lerobot`) | `select_action` (hàng đợi), `sample_actions` (khử nhiễu `num_steps` lần), `prepare_images` (bù camera thiếu) |
| Notebook Colab mục 8 | Bản thật: ~2 000 ván, 20 000 step, chấm 774 seed, so với ACT 84,8 % |